# Mustache Utilities API Reference

Developer-facing statements defined in `libs/core/langchain_core/utils/mustache.py`.

# `Scopes`

Stack of Mustache rendering scopes. Each scope is either a mapping or the falsy sentinel `False` or `0`.

```python
Scopes = list[Literal[False, 0] | Mapping[str, Any]]
```

`TypeAlias` is imported only under `TYPE_CHECKING`, but `Scopes` itself is defined and available at runtime.

---

# `ChevronError: SyntaxError`

Raised when a Mustache template contains invalid syntax, including empty or unclosed tags, malformed delimiter tags, mismatched section endings, closing unopened sections, or reaching the end of the template with an open section.

---

# `tokenize`

Tokenizes a Mustache template lazily.

```python
tokenize(
    template: str, # Mustache template to tokenize
    def_ldel: str = "{{", # Initial left delimiter
    def_rdel: str = "}}", # Initial right delimiter
) -> Iterator[tuple[str, str]] # Pairs containing a token type and token value
```

The iterator can produce literal, variable, no-escape, section, inverted-section, end, partial, and set-delimiter tokens. Comments are not yielded. Standalone tags remove their surrounding indentation and line content according to the parser's standalone-tag handling.

Raises `ChevronError` when the template contains invalid tag or section syntax.

---

# `render`

Renders a Mustache template with data scopes and inline partials.

```python
render(
    template: str | list[tuple[str, str]] = "", # Template string or pre-tokenized template
    data: Mapping[str, Any] = EMPTY_DICT, # Initial data scope
    partials_dict: Mapping[str, str] = EMPTY_DICT, # Mapping of partial names to template strings
    padding: str = "", # Prefix applied after newlines while rendering partial content
    def_ldel: str = "{{", # Initial left delimiter
    def_rdel: str = "}}", # Initial right delimiter
    scopes: Scopes | None = None, # Explicit scope stack, or None to create one from data
    warn: bool = False, # Whether to log warnings for missing substitutions
    keep: bool = False, # Whether to preserve missing variable tags
) -> str # Rendered template
```

String templates are tokenized unless cached internally; a token list is consumed directly. Variables are HTML-escaped, while `{{& name}}` and triple-brace tags are inserted without escaping.

Sections support mappings, sequences, iterators, generators, scalar truthy values, and callable lambdas. Inverted sections render when their resolved value is falsy. Partials are loaded only from `partials_dict`; missing partials render as empty strings.

Dotted lookup traverses dictionaries and numeric indexes in lists or tuples. Arbitrary object-attribute traversal is rejected. Missing values render as an empty string unless `keep=True`, and can be logged when `warn=True`.

When an explicit `scopes` list is supplied, it is used directly and is modified as sections are entered and exited.

Raises `ChevronError` when tokenization encounters invalid Mustache syntax.

In [ ]:
# 1. Render a basic template
from langchain_core.utils.mustache import render # Import the Mustache rendering function


template = "Hello, {{name}}! Welcome to {{course}}." # Create a template with variable placeholders

data = { # Create the template data
    "name": "Saad", # Provide the name value
    "course": "LangChain", # Provide the course value
} # Finish creating the data dictionary

result = render( # Render the Mustache template
    template=template, # Provide the template string
    data=data, # Provide values for the placeholders
) # Finish rendering the template

print(result) # Display the rendered output

In [ ]:
# 2. Render a list using a section
from langchain_core.utils.mustache import render # Import the Mustache rendering function


template = """ # Create a template containing a list section
Skills:
{{#skills}}
- {{name}}
{{/skills}}
""" # Finish creating the template

data = { # Create data containing a list of skills
    "skills": [ # Begin the skill list
        {"name": "Python"}, # Add the first skill
        {"name": "SQL"}, # Add the second skill
        {"name": "LangChain"}, # Add the third skill
    ] # Finish the skill list
} # Finish creating the data dictionary

result = render(template=template, data=data) # Render the section once for each list item

print(result) # Display the rendered skill list

In [ ]:
# 3. Use normal and inverted sections
from langchain_core.utils.mustache import render # Import the Mustache renderer


template = """ # Create a template with normal and inverted sections
{{#is_logged_in}}
Welcome back, {{name}}!
{{/is_logged_in}}

{{^is_logged_in}}
Please log in to continue.
{{/is_logged_in}}
""" # Finish creating the template

logged_in_data = { # Create data for a logged-in user
    "is_logged_in": True, # Enable the normal section
    "name": "Saad", # Provide the user's name
} # Finish creating the logged-in data

logged_out_data = { # Create data for a logged-out user
    "is_logged_in": False, # Enable the inverted section
} # Finish creating the logged-out data

print(render(template=template, data=logged_in_data)) # Render the logged-in version
print(render(template=template, data=logged_out_data)) # Render the logged-out version

In [ ]:
# 4. Use dotted variable lookup
from langchain_core.utils.mustache import render # Import the Mustache renderer


template = "Employee: {{employee.name}}, Department: {{employee.department}}" # Create a template using dotted keys

data = { # Create nested data
    "employee": { # Create the nested employee mapping
        "name": "Saad", # Store the employee name
        "department": "Engineering", # Store the department
    } # Finish the employee mapping
} # Finish creating the data dictionary

result = render(template=template, data=data) # Resolve the dotted dictionary values

print(result) # Display the rendered employee details

In [ ]:
# 5. Escaped and unescaped values
from langchain_core.utils.mustache import render # Import the Mustache renderer


template = """ # Create a template showing escaped and raw rendering
Escaped: {{html}}
Raw: {{{html}}}
Raw using &: {{& html}}
""" # Finish creating the template

data = { # Create data containing HTML
    "html": "<strong>LangChain</strong>", # Store an HTML-formatted value
} # Finish creating the data dictionary

result = render(template=template, data=data) # Render escaped and unescaped versions

print(result) # Display the rendering difference

In [ ]:
# 6. Render a partial template
from langchain_core.utils.mustache import render # Import the Mustache renderer


template = """ # Create the main template
User details:
{{> user_card}}
""" # Finish creating the main template

partials = { # Create the available partial templates
    "user_card": "Name: {{name}}\nRole: {{role}}\n", # Define the reusable user-card partial
} # Finish creating the partial dictionary

data = { # Create the partial's data
    "name": "Saad", # Provide the user's name
    "role": "Python Developer", # Provide the user's role
} # Finish creating the data dictionary

result = render( # Render the main template and its partial
    template=template, # Provide the main template
    data=data, # Provide the rendering data
    partials_dict=partials, # Provide the available partial templates
) # Finish rendering the template

print(result) # Display the rendered partial

In [ ]:
# 7. Preserve missing variables
from langchain_core.utils.mustache import render # Import the Mustache renderer


template = "Hello, {{name}}! Your role is {{role}}." # Create a template containing a missing variable

data = { # Create incomplete template data
    "name": "Saad", # Provide only the name value
} # Finish creating the data dictionary

default_result = render(template=template, data=data) # Replace the missing role with an empty string
kept_result = render(template=template, data=data, keep=True) # Preserve the unresolved role tag

print("Default:", default_result) # Display the default missing-value behavior
print("Keep:", kept_result) # Display the preserved Mustache placeholder

In [ ]:
# 8. Inspect tokens using tokenize()
from langchain_core.utils.mustache import tokenize # Import the lazy Mustache tokenizer


template = "Hello {{name}}! {{#active}}Account active{{/active}}" # Create a template containing variables and a section

tokens = list(tokenize(template)) # Convert the lazy token iterator into a list

for token_type, token_value in tokens: # Iterate through the generated tokens
    print(token_type, ":", repr(token_value)) # Display each token type and value

In [ ]:
# 9. Handle invalid Mustache syntax
from langchain_core.utils.mustache import ChevronError, render # Import the renderer and syntax exception


invalid_template = "{{#user}}Hello, {{name}}" # Create a template with an unclosed section

try: # Begin exception handling
    result = render( # Try to render the invalid template
        template=invalid_template, # Provide the malformed template
        data={"user": True, "name": "Saad"}, # Provide template data
    ) # Finish the rendering call

except ChevronError as error: # Catch the Mustache syntax error
    print("Template error:") # Display an error heading
    print(error) # Display the syntax error message